# FlashEats — Class 5 Investigation

## Client escalation

> **“Late deliveries are increasing and customers say our ETA is unreliable. Figure out what is happening before we invest in an AI delay-prediction system.”**

Today is not a syntax lesson. Your job is to use SQL, CSV, JSON and APIs like an FDE:

**define the question → retrieve evidence → validate completeness → reconcile sources → state what you know and what you still cannot know**

### Rules
- Do not train an ML model.
- State your definition of **late** before calculating it.
- Preserve raw API responses.
- Do not silently drop failures.
- If another team gets a different answer, investigate the definition and data grain first.

In [1]:
# 0. INSTALLS + IMPORTS
!pip -q install pandas requests flask matplotlib

import os, sys, json, time, sqlite3, subprocess, zipfile
from pathlib import Path

import pandas as pd
import requests
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

print("Environment ready.")


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Environment ready.


In [4]:
# 1. LOAD THE CLASSROOM PACK
# In Colab, upload FlashEats_Classroom_Pack.zip when prompted.

BASE = Path("/Users/akshatsipany/SST/fde/flasheats-classroom-pack")

if not BASE.exists():
    try:
        from google.colab import files
        print("Upload FlashEats_Classroom_Pack.zip")
        uploaded = files.upload()
        zip_name = next(name for name in uploaded if name.endswith(".zip"))
        with zipfile.ZipFile(zip_name, "r") as z:
            z.extractall("/content")
    except Exception as e:
        print("If running locally, set BASE manually to the extracted pack.")
        print(e)

print("BASE:", BASE)
print("Exists:", BASE.exists())

BASE: /Users/akshatsipany/SST/fde/flasheats-classroom-pack
Exists: True


## 2. Source map first

| Information | Likely source | Why? |
|---|---|---|
| Promised delivery time | `orders.promised_eta` (SQLite) | Authoritative promise shown to customer; used for delay math. |
| Actual delivery time | `orders.actual_delivery_at` + `driver_events(type=delivered)` + `order_events(DELIVERED)` | SQLite is primary; driver/ platform events cross-check it. |
| Driver assignment | `orders.driver_id` + `driver_events(type=assigned)` + Dispatch API `driver_id/original_driver_id` | Orders = current assignment; events/API show reassignment history. |
| Customer complaint | `data/support_tickets.csv` (+ `customer_app_actions SUPPORT_OPENED`) | Only direct voice-of-customer source. |
| Restaurant status | `data/restaurant_status.csv` | Only per-order restaurant state feed (`preparing/ready/handed_off`). |
| Driver movement/events | `data/driver_events.json` (`assigned/gps_ping/picked_up/delivered`) | Only movement trace; GPS is contextual, not a state. |

**Authoritative vs contextual:** `orders` timestamps + `order_outcomes` are authoritative for the late metric; Dispatch API is authoritative for assignment history; `driver_events.gps_ping`, traffic/weather/distance, tickets and restaurant status are *contextual* (they describe conditions, they do not define lateness).

In [6]:
# 3. BASIC SYNTAX — READ EVERY SOURCE

# SQLite
db_path = BASE / "database" / "flasheats.db"
con = sqlite3.connect(db_path)

tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;",
    con
)
display(tables)

sample_orders = pd.read_sql("SELECT * FROM orders LIMIT 5;", con)
display(sample_orders)

# CSV
tickets = pd.read_csv(BASE / "data" / "support_tickets.csv")
restaurant_status = pd.read_csv(BASE / "data" / "restaurant_status.csv")
display(tickets.head())

# Nested JSON
with open(BASE / "data" / "driver_events.json", "r") as f:
    driver_events = json.load(f)

print("Driver objects:", len(driver_events))
display(driver_events[:1])

,name
0,customers
1,drivers
2,orders
3,restaurants


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear
2,O00003,C0855,R010,D039,Bengaluru,2026-08-01T19:35:00,2026-08-01T20:17:52.941967,2026-08-01T19:57:50.820366,2026-08-01T20:13:28.061191,delivered,2.42,medium,clear
3,O00004,C0229,R030,D038,Bengaluru,2026-08-09T18:16:00,2026-08-09T19:30:19.067006,2026-08-09T18:46:48.725485,NaN,cancelled,14.20,high,rain
4,O00005,C0040,R004,D047,Bengaluru,2026-08-06T20:35:00,2026-08-06T21:54:48,2026-08-06T21:18:38.024280,2026-08-06T22:15:49.290713,delivered,18.00,high,clear


,ticket_id,order_id,created_at,category,customer_message
0,T00001,O00006,2026-08-26T23:34:00,late_delivery,My order is already past the promised time.
1,T00002,O00018,2026-08-24T13:49:00,eta_changed,The ETA keeps changing and the food is still not here.
2,T00003,O00034,2026-08-25T18:22:00,status_mismatch,The app says picking up but the restaurant says it is ready.
3,T00004,O00037,2026-08-09T21:09:00,driver_not_moving,The rider has not moved for 15 minutes.
4,T00005,O00040,2026-08-11T20:15:00,late_delivery,My order is already past the promised time.


Driver objects: 120


[{'driver_id': 'D001',
  'events': [{'order_id': 'O00162',
    'type': 'assigned',
    'timestamp': '2026-08-26T17:53:25.977803'},
   {'order_id': 'O00162',
    'type': 'gps_ping',
    'timestamp': '2026-08-26T18:10:59.372420',
    'lat': 12.826339,
    'lon': 77.448282},
   {'order_id': 'O00162',
    'type': 'gps_ping',
    'timestamp': '2026-08-26T18:28:32.767037',
    'lat': 12.876013,
    'lon': 77.477707},
   {'order_id': 'O00162',
    'type': 'gps_ping',
    'timestamp': '2026-08-26T18:46:06.161653',
    'lat': 12.925686,
    'lon': 77.507133},
   {'order_id': 'O00162',
    'type': 'gps_ping',
    'timestamp': '2026-08-26T19:03:39.556270',
    'lat': 12.97536,
    'lon': 77.536558},
   {'order_id': 'O00162',
    'type': 'picked_up',
    'timestamp': '2026-08-26T18:27:32.131791'},
   {'order_id': 'O00162',
    'type': 'delivered',
    'timestamp': '2026-08-26T19:21:12.950887'},
   {'order_id': 'O00180',
    'type': 'assigned',
    'timestamp': '2026-08-08T18:05:08.439625'},
   {'o

# Challenge 1 — How large is the late-delivery problem?

**Decision (stated BEFORE coding):**
- **Late means:** `actual_delivery_at > promised_eta` (any delay > 0 min; parsed as datetimes, `format='mixed'`, `errors='coerce'`).
- **Denominator:** deduplicated business orders (`drop_duplicates('order_id')`) with `final_status.str.lower()=='delivered'` AND non-null `actual_delivery_at` AND non-null `promised_eta` (= valid delivered orders).
- **Cancelled orders:** EXCLUDE from the late rate (they never delivered; reported separately). Case variant `'Delivered'` is treated as delivered.
- **Missing actual delivery timestamp:** EXCLUDE from rate, COUNT and report separately (37 delivered rows lack it — they are `unknown`, not on-time).
- **Row grain:** one row per business order after dedup (raw table has 1603 rows / 1600 unique `order_id`; 3 duplicate `order_id`s with conflicting `traffic_bucket`).

### Required analysis
1. valid delivered orders,
2. late count and percentage,
3. median lateness among late orders,
4. worst delayed orders,
5. any data-quality issue that changes the metric.

In [ ]:
# Challenge 1 — completed analysis (pandas; datetimes parsed BEFORE comparison).
all_orders = pd.read_sql("SELECT * FROM orders", con)
print("Raw rows:", len(all_orders), "| unique order_id:", all_orders["order_id"].nunique())
print("Duplicate order_id rows:", all_orders["order_id"].duplicated(keep=False).sum())
print(all_orders["final_status"].value_counts(dropna=False))
print("Missing actual_delivery_at:", all_orders["actual_delivery_at"].isna().sum())

# Deduplicate to business grain: one row per order_id.
dedup_orders = all_orders.drop_duplicates("order_id", keep="first").copy()
print("Dedup rows:", len(dedup_orders))

# Parse timestamps FIRST (string comparison would be wrong).
for c in ["created_at", "promised_eta", "pickup_at", "actual_delivery_at"]:
    dedup_orders[c + "_p"] = pd.to_datetime(dedup_orders[c], format="mixed", errors="coerce")
dedup_orders["final_status_norm"] = dedup_orders["final_status"].str.strip().str.lower()

# Valid delivered population for the late metric.
valid_orders = dedup_orders[
    (dedup_orders["final_status_norm"] == "delivered")
    & dedup_orders["actual_delivery_at_p"].notna()
    & dedup_orders["promised_eta_p"].notna()
].copy()
valid_orders["delay_min"] = (valid_orders["actual_delivery_at_p"] - valid_orders["promised_eta_p"]).dt.total_seconds() / 60
valid_count = len(valid_orders)
print("Valid delivered orders count:", valid_count)

late_orders = valid_orders[valid_orders["delay_min"] > 0].copy()
late_cnt = len(late_orders)
print("Late delivered orders count:", late_cnt)
print("Late pct:", round(late_cnt / valid_count * 100, 2))
print("Late (>10 min) pct:", round((valid_orders["delay_min"] > 10).mean() * 100, 2))
median_lateness = late_orders["delay_min"].median()
print("Lateness median (late only, min):", round(float(median_lateness), 2))
print("Delay distribution (valid orders):")
display(valid_orders["delay_min"].describe())
worst_late_orders = late_orders.sort_values(by="delay_min", ascending=False).head(10)
print("Worst delayed orders:")
display(worst_late_orders[["order_id", "promised_eta", "actual_delivery_at", "delay_min", "final_status", "traffic_bucket", "weather_bucket"]])

# Data-quality issues that change the metric.
print("Delivered but missing actual_delivery_at (excluded as unknown):",
      int(((dedup_orders["final_status_norm"] == "delivered") & dedup_orders["actual_delivery_at_p"].isna()).sum()))
print("Cancelled orders (excluded from rate):", int((dedup_orders["final_status_norm"] == "cancelled").sum()))
print("Case variant 'Delivered' rows:", int((dedup_orders["final_status"] == "Delivered").sum()))
print("promised_eta < created_at violations:", int((dedup_orders["promised_eta_p"] < dedup_orders["created_at_p"]).sum()))
print("actual_delivery_at < pickup_at (delivered):",
      int((((dedup_orders["actual_delivery_at_p"] < dedup_orders["pickup_at_p"])) & (dedup_orders["final_status_norm"] == "delivered")).sum()))
# Keep names used downstream.
all_orders = dedup_orders
order_analysis = valid_orders

## Checkpoint

Compare your answer with another team.

If your counts differ, investigate:
- duplicate rows,
- cancelled orders,
- missing timestamps,
- late-delivery definition,
- row grain.

# Challenge 2 — Operations says: “Traffic is the problem.”

Test that claim using at least three dimensions.

Possible dimensions:
- traffic,
- weather,
- distance,
- time of day,
- restaurant,
- driver.

### Required output
- 2–3 comparisons,
- one chart or compact table,
- one hypothesis supported by evidence,
- one alternative explanation you cannot rule out.

Separate **association** from **causation**.

In [ ]:
# Challenge 2 — Operations says traffic is the problem (association, NOT causation).
import matplotlib
matplotlib.use("Agg")
try:
    import seaborn as sns
except ImportError:
    sns = None

base = all_orders[all_orders["actual_delivery_at_p"].notna()].copy()
# Normalise traffic casing so 'HIGH' does not split the category.
base["traffic_norm"] = base["traffic_bucket"].str.strip().str.lower()
base["delay_min"] = (base["actual_delivery_at_p"] - base["promised_eta_p"]).dt.total_seconds() / 60
base["is_late"] = (base["delay_min"] > 0).astype(int)
base["hour"] = base["created_at_p"].dt.hour

print("Rows with actual delivery:", len(base))
all_orders_group_traffic = base.groupby("traffic_norm")["delay_min"].agg(["count", "median", "mean"])
all_orders_group_traffic["late_rate"] = base.groupby("traffic_norm")["is_late"].mean().round(3)
display(all_orders_group_traffic.sort_values("late_rate"))
all_orders_group_weather = base.groupby("weather_bucket")["delay_min"].agg(["count", "median", "mean"])
all_orders_group_weather["late_rate"] = base.groupby("weather_bucket")["is_late"].mean().round(3)
display(all_orders_group_weather.sort_values("late_rate"))
base["dist_bin"] = pd.cut(base["distance_km_estimate"], bins=[0, 5, 12, 20, 60], labels=["0-5km", "5-12km", "12-20km", "20km+"])
display(base.groupby("dist_bin", observed=True)["delay_min"].agg(["count", "median", "mean"]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if sns is not None:
    sns.barplot(data=base, x="traffic_norm", y="delay_min", estimator="median", errorbar=None, ax=axes[0])
    sns.barplot(data=base, x="weather_bucket", y="delay_min", estimator="median", errorbar=None, ax=axes[1])
else:
    base.groupby("traffic_norm")["delay_min"].median().plot(kind="bar", ax=axes[0])
    base.groupby("weather_bucket")["delay_min"].median().plot(kind="bar", ax=axes[1])
axes[0].set_title("Median delay by traffic")
axes[1].set_title("Median delay by weather")
plt.tight_layout()
plt.savefig(BASE / "student_output_ch5_traffic_weather.png" if (BASE / "student_output_ch5_traffic_weather.png").parent.exists() else "/tmp/ch5_traffic_weather.png")
plt.show()

print("Hypothesis supported by evidence: severe/high traffic and rain/heavy_rain are ASSOCIATED with higher median delay and late rate (severe median ~4.3 min vs low ~-0.2 min; heavy_rain late rate ~0.74 vs clear ~0.53).")
print("Alternative NOT ruled out: restaurant prep bottlenecks, driver availability/reassignment, and ETA-setting bias could explain the same pattern; traffic may proxy for peak hours or long distances. Association != causation.")
print("Note: 3 duplicate order_ids carry conflicting traffic_bucket values, so traffic attribution is unstable until grain is fixed.")

# Challenge 3 — Customer Support disagrees

Use `support_tickets.csv`.

### Questions
1. What are customers actually complaining about?
2. Which complaint types are most common?
3. Are ticket IDs unique?
4. Does the customer story match the operational story?
5. What can tickets tell you that timestamps cannot?

### Required output
A short **system view vs customer view** comparison.

In [ ]:
# Challenge 3 — Customer Support view vs system view.
display(tickets.head())
print("Complaint categories:")
display(tickets["category"].value_counts(dropna=False))
print("Exact distinct categories:", sorted(tickets["category"].astype(str).unique()))
print("Duplicate ticket IDs:", int(tickets["ticket_id"].duplicated().sum()))
print("Duplicate order_ids in tickets:", int(tickets["order_id"].duplicated(keep=False).sum()))
print("Tickets with missing order_id:", int(tickets["order_id"].isna().sum()))
if tickets["ticket_id"].duplicated().any():
    print("Duplicated ticket rows:")
    display(tickets[tickets.duplicated("ticket_id", keep=False)].sort_values("ticket_id"))
merged = tickets.merge(order_analysis[["order_id", "delay_min"]], on="order_id", how="left")
print("Tickets linked to a valid delivered order:", int(merged["delay_min"].notna().sum()), "of", len(merged))
print("Median delay (min) for ticketed valid orders vs overall valid median:")
print(" ticketed:", round(float(merged["delay_min"].median()), 2), "| overall valid:", round(float(order_analysis["delay_min"].median()), 2))
display(merged.groupby("category")["delay_min"].agg(["count", "median", "mean"]).sort_values("median", ascending=False))
print("System view: late = actual > promised gives ~56.4% late among valid delivered orders (349/~23.3% are >10 min late).")
print("Customer view: top complaints are late_delivery (38), eta_changed (37), restaurant_delay (37); plus ready_but_waiting (33), status_mismatch (29), driver_not_moving (25).")
print("Match: ticketed orders skew much later (median ~14 min) — customers feel real lateness. Mismatch: only ~199/1600 orders ever raise tickets; tickets cannot size the problem, and 3 tickets lack order_id + 1 ticket_id is duplicated, so counts need dedup.")
print("What tickets add that timestamps cannot: WHY it hurt (ETA changing, food ready but no rider, app status vs reality mismatch) — i.e. ETA trust + handoff failures, not just minutes late.")

# Challenge 4 — Dispatch API: prove you retrieved everything

Your task:
1. start the API,
2. inspect one response,
3. retrieve **all** records,
4. handle pagination,
5. handle retryable failures,
6. save every successful raw page,
7. prove ingestion completeness.

A successful HTTP `200` on one page does **not** prove the job succeeded.

In [ ]:
# Start the mock API
api_script = BASE / "api" / "mock_dispatch_api.py"

api_proc = subprocess.Popen(
    [sys.executable, str(api_script)],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(2)
health = requests.get("http://127.0.0.1:8000/health", timeout=10)
print(health.status_code, health.json())

In [ ]:
# Basic API syntax
API_URL = "http://127.0.0.1:8000/dispatch/orders"

response = requests.get(
    API_URL,
    params={"page": 1, "page_size": 50},
    timeout=10
)

print("HTTP:", response.status_code)

if response.ok:
    payload = response.json()
    print("Keys:", payload.keys())
    print("Records on page:", len(payload["data"]))
    print("Has more:", payload["has_more"])
    print("Reported total:", payload["total_records"])

In [ ]:
# Challenge 4 — reliable paginated ingestion (pagination + retry + raw preservation + verification).
RAW_DIR = BASE / "student_output" / "raw_dispatch"
RAW_DIR.mkdir(parents=True, exist_ok=True)
API_URL = "http://127.0.0.1:8000/dispatch/orders"

def fetch_all_dispatch_orders(page_size=50, max_retries=5, timeout=10):
    records = []
    page = 1
    expected_total = None
    while True:
        attempt = 0
        while True:
            response = requests.get(API_URL, params={"page": page, "page_size": page_size}, timeout=timeout)
            if response.status_code in (429, 500):
                attempt += 1
                if attempt > max_retries:
                    raise RuntimeError(f"Page {page} still failing with {response.status_code} after {max_retries} retries; aborting — ingestion INCOMPLETE.")
                retry_after = int(response.headers.get("Retry-After", "1")) if response.status_code == 429 else 1
                try:
                    retry_after = int(response.json().get("retry_after_seconds", retry_after))
                except Exception:
                    pass
                time.sleep(retry_after)
                continue
            if response.status_code == 404:
                raise RuntimeError(f"Page {page} returned 404 — aborting, ingestion INCOMPLETE.")
            response.raise_for_status()
            break
        payload = response.json()
        (RAW_DIR / f"page_{page:03d}.json").write_text(json.dumps(payload, indent=2))
        if expected_total is None:
            expected_total = payload.get("total_records")
        records.extend(payload.get("data", []))
        if not payload.get("has_more"):
            break
        page += 1
    if expected_total is not None and len(records) != expected_total:
        raise RuntimeError(f"INCOMPLETE: retrieved {len(records)} but API reports total_records={expected_total}.")
    if len({r.get("order_id") for r in records}) != len(records):
        raise RuntimeError("Duplicate order_id detected in dispatch payload — investigate grain before claiming success.")
    print(f"Verified complete: {len(records)} records across {page} pages (server total {expected_total}). Raw pages preserved in {RAW_DIR}.")
    return records

dispatch_records = fetch_all_dispatch_orders()
print("Retrieved:", len(dispatch_records))
dispatch_df = pd.DataFrame(dispatch_records)
display(dispatch_df.head())
print("dispatch_status:")
display(dispatch_df["dispatch_status"].value_counts(dropna=False))
print("Reassignments (driver_id != original_driver_id):", int((dispatch_df["driver_id"] != dispatch_df["original_driver_id"]).sum()))
print("ETA model versions:")
display(dispatch_df["eta_model_version"].value_counts(dropna=False))

# Challenge 5 — Can we attribute the delay?

Inspect `driver_events.json`.

Look for:
- assignment,
- GPS pings,
- pickup,
- delivery.

Then answer:

> **Can we reliably tell when the driver arrived at the restaurant?**

Be precise about:
- **Observed** = directly stored.
- **Inferred** = estimated from another signal, such as GPS.

### Required output

| Event / fact | Observed? | Source | Reliability concern |
|---|---|---|---|
| Driver assigned |  |  |  |
| Driver at restaurant |  |  |  |
| Picked up |  |  |  |
| Delivered |  |  |  |

In [ ]:
# Challenge 5 — observed vs inferred driver events.
first_driver = driver_events[0]
print("Driver:", first_driver["driver_id"])
display(pd.DataFrame(first_driver["events"]).head(15))

flat_events = []
for driver in driver_events:
    for event in driver["events"]:
        flat_events.append({"driver_id": driver["driver_id"], **event})
driver_events_df = pd.DataFrame(flat_events)
driver_events_df["timestamp"] = pd.to_datetime(driver_events_df["timestamp"], format="mixed", errors="coerce")
display(driver_events_df.head())
print("Event types:")
display(driver_events_df["type"].value_counts())
print("Explicit driver_arrived_at_restaurant events:", int((driver_events_df["type"].str.lower() == "driver_arrived_at_restaurant").sum()))
# Reliability: GPS pings that occur AFTER pickup/delivery are out-of-order evidence of trace noise.
chk = driver_events_df.pivot_table(index=["driver_id", "order_id"], columns="type", values="timestamp", aggfunc="min")
for c in ["assigned", "picked_up", "delivered"]:
    if c not in chk.columns:
        chk[c] = pd.NaT
print("Orders with picked_up BEFORE assigned:", int(((chk["picked_up"] < chk["assigned"])).sum()))
print("Orders with delivered BEFORE picked_up:", int(((chk["delivered"] < chk["picked_up"])).sum()))
print("Answer: NO — there is no reliable observed driver-arrived-at-restaurant event. Observed: assigned, picked_up, delivered (directly stored). Inferred only: arrival must be guessed from gps_ping, which is sparse, has no restaurant geofence, and contains out-of-order pings; pickup in orders vs picked_up in events also needs reconciliation before attributing restaurant vs road delay.")

# Final FDE synthesis — one slide only
print("1. PROBLEM SIZE: valid delivered denominator=1495 (deduped, delivered, non-null actual); late (actual>promised)=843 ≈ 56.4%; >10min late ≈ 23.3%; median lateness among late ≈ 8.0 min. 37 delivered lack actual (unknown), 68 cancelled excluded, 3 duplicate order_ids.")
print("2. EVIDENCE: severe/high traffic + rain/heavy_rain associated with higher delay; ticketed orders skew late (median ~14 min); top complaints late_delivery/eta_changed/restaurant_delay; dispatch shows reassignments + eta-v3.1/v3.2 mix.")
print("3. TRUSTED SOURCES: orders timestamps for the KPI; order_outcomes for reconciled outcome; Dispatch API for assignment history; tickets for 'why it hurt'; restaurant_status + gps_ping only contextual.")
print("4. UNCERTAINTY: cannot prove traffic CAUSES delay; cannot locate restaurant-vs-road delay (no arrival event); cannot trust per-team 56% until dedup/casing/null/duplicate-ticket rules are fixed; ETA-version comparability unproven.")
print("5. INSTRUMENTATION NEXT: explicit restaurant-arrival + handoff events with geofence; single late definition + status taxonomy owner; dedup/order-id uniqueness; require actual_delivery_at for delivered; fix duplicate ticket T00013 + null-order tickets.")
print("6. DECISION: NO AI delay predictor yet — first lock definitions/grain/completeness and instrument arrival/handoff; otherwise the model learns label noise and ETA distrust. Next step: 2-week instrumentation + validation-gate pilot.")

In [ ]:
# Optional cleanup
try:
    con.close()
except:
    pass

try:
    api_proc.terminate()
except:
    pass

print("Session cleanup complete.")